# 02. Carga de Snapshots CGR (INFOBRAS) por Rango de Fechas

Carga los archivos `DataSet-Obras-Publicas DD-MM-YYYY.xlsx` de `data/raw/cgr` cuya **fecha de snapshot** esté dentro de un rango, aplicando el mapping `docs/datamappings/stg__dataset_obras_publicas__to__sa_obras_publicas_infobras.yaml`, y los consolida en un único `DataFrame` de pandas.

**Parámetros de entrada**: `FECHA_INICIO` y `FECHA_FIN` (formato `YYYYMMDD`, ambos inclusive). Ejemplo `("20260101", "20260301")` carga: `01-01-2026`, `02-01-2026`, `01-02-2026` y `01-03-2026`.

**Qué hace el mapping**:
- Resuelve encabezados repetidos (RUC, Monto del contrato, etc.) **por posición** (`source_occurrence`).
- Renombra a `snake_case` destino y aplica las transformaciones (`TRF_*`): texto, entero estricto, decimal, fecha *dayfirst* y booleano Sí/No.
- Agrega las columnas de auditoría/linaje (`fecha_foto`, `nombre_archivo`, `fecha_creacion`, `usuario_creacion`) **solo si tienen `active: true`** en `audit_columns`; las que tienen `active: false` se ignoran y no se cargan.
- Columna de origen ausente → se carga `NULL` con warning; columna no mapeada → se omite con warning.

**Opcional**: exportar el dataset a CSV en `data/stg` (`EXPORTAR_CSV = True`).

## 1. Parámetros

In [16]:
# ==============================================================================
# PARÁMETROS DE ENTRADA
# ==============================================================================
FECHA_INICIO = "20260201"   # YYYYMMDD (inclusive)
FECHA_FIN = "20260401"      # YYYYMMDD (inclusive)

FILA_CABECERA = None        # None = autodetectar (busca la fila con "Código INFOBRAS"); o un entero (1-based)
EXPORTAR_CSV = True         # True para exportar el dataset consolidado a data/stg
NOMBRE_CSV = None           # None = stg_obras_publicas_infobras_<inicio>_<fin>.csv
# ==============================================================================

## 2. Configuración, rutas y lectura del mapping

In [17]:
import re
import logging
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S", force=True)
log = logging.getLogger("carga_cgr")


def obtener_raiz_proyecto() -> Path:
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "docs" / "datamappings").exists():
            return carpeta
    return actual


PROJECT_ROOT = obtener_raiz_proyecto()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "cgr"
STG_DIR = PROJECT_ROOT / "data" / "stg"
MAPPING_PATH = PROJECT_ROOT / "docs" / "datamappings" / "stg__dataset_obras_publicas__to__sa_obras_publicas_infobras.yaml"

with open(MAPPING_PATH, encoding="utf-8") as f:
    mapping = yaml.safe_load(f)

print(f"Raíz proyecto : {PROJECT_ROOT}")
print(f"Origen (raw)  : {RAW_DIR}")
print(f"Destino (stg) : {STG_DIR}")
print(f"Mapping       : {MAPPING_PATH.name} (v{mapping['mapping']['version']})")

Raíz proyecto : C:\ws\formacion\uni-mia\mia-tesis-opa
Origen (raw)  : C:\ws\formacion\uni-mia\mia-tesis-opa\data\raw\cgr
Destino (stg) : C:\ws\formacion\uni-mia\mia-tesis-opa\data\stg
Mapping       : stg__dataset_obras_publicas__to__sa_obras_publicas_infobras.yaml (v1.0.0)


## 3. Mapping de columnas

Se construye una tabla `(encabezado normalizado, ocurrencia) → columna destino, tipo, transformación`. El encabezado se normaliza (minúsculas, espacios colapsados y sin `:` final) para tolerar variaciones como `"Nombre o razón social de la empresa o consorcio:"`.

In [18]:
def norm_header(texto) -> str:
    t = re.sub(r"\s+", " ", str(texto)).strip().lower()
    return t.rstrip(":").strip()


columnas_mapping = []  # lista ordenada según el YAML
for grupo in mapping["column_groups"]:
    for col in grupo["columns"]:
        columnas_mapping.append({
            "grupo": grupo["group"],
            "source": col["source"],
            "source_norm": norm_header(col["source"]),
            "occurrence": col.get("source_occurrence", 1),
            "target": col["target"],
            "type": col["type"],
            "transform": col["transform"],
        })

df_map = pd.DataFrame(columnas_mapping)
LOOKUP = {(r.source_norm, r.occurrence): r for r in df_map.itertuples()}
TARGET_COLS = df_map["target"].tolist()

import getpass

# Columnas de auditoría/linaje: se agregan solo si `active: true` en el mapping
AUDIT_GENERATORS = {
    "fecha_foto": lambda ctx: ctx["fecha_foto"],
    "nombre_archivo": lambda ctx: ctx["nombre_archivo"],
    "fecha_creacion": lambda ctx: pd.Timestamp.now().floor("s"),
    "usuario_creacion": lambda ctx: getpass.getuser(),
}
AUDIT_ACTIVE = [a["target"] for a in mapping["audit_columns"] if a.get("active", True)]
AUDIT_IGNORED = [a["target"] for a in mapping["audit_columns"] if not a.get("active", True)]
sin_generador = set(AUDIT_ACTIVE) - set(AUDIT_GENERATORS)
assert not sin_generador, f"Columnas de auditoría activas sin generador: {sin_generador}"

print(f"Columnas mapeadas: {len(df_map)}")
print(f"Auditoría activa  : {AUDIT_ACTIVE}")
print(f"Auditoría ignorada: {AUDIT_IGNORED}")
df_map["type"].value_counts().to_frame("columnas")

Columnas mapeadas: 113
Auditoría activa  : ['fecha_foto']
Auditoría ignorada: ['nombre_archivo', 'fecha_creacion', 'usuario_creacion']


,columnas
type,
string,48
decimal,21
integer,18
date,18
boolean,8


## 4. Transformaciones (`TRF_*`)

In [19]:
BOOL_TRUE = {v.lower() for v in map(str, mapping["conventions"]["boolean_true_values"])}
BOOL_FALSE = {v.lower() for v in map(str, mapping["conventions"]["boolean_false_values"])}
INT32_MIN, INT32_MAX = -(2**31), 2**31 - 1


def _a_texto(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    return s.mask(s.isin(["", "nan", "NaN", "None", "NaT"]))


def trf_none(s):
    return _a_texto(s)


def trf_force_string(s):
    return _a_texto(s)


def trf_amount_space_to_decimal(s):
    """'S/1 002 000 00' -> 1002000.00 ; '1205065 23' -> 1205065.23 (último espacio = decimal)."""
    t = _a_texto(s)
    t = t.str.replace(r"(?i)^s/\s*", "", regex=True)
    t = t.str.replace(r"\s+(\d+)$", r".\1", regex=True)   # último espacio -> separador decimal
    t = t.str.replace(r"[\s,]", "", regex=True)           # miles
    return pd.to_numeric(t, errors="coerce").astype("Float64")


def trf_date_dayfirst(s):
    t = _a_texto(s)
    return pd.to_datetime(t, errors="coerce", dayfirst=True, format="mixed")


def trf_boolean_si_no(s):
    t = _a_texto(s).str.lower()
    out = pd.Series(pd.NA, index=s.index, dtype="boolean")
    out[t.isin(BOOL_TRUE).fillna(False)] = True
    out[t.isin(BOOL_FALSE).fillna(False)] = False
    return out


def trf_integer_strict(s):
    t = _a_texto(s).str.replace(r"[\s,]", "", regex=True)
    n = pd.to_numeric(t, errors="coerce")
    n = n.where((n % 1 == 0) & n.between(INT32_MIN, INT32_MAX))
    return n.astype("Int64")


TRANSFORMS = {
    "TRF_NONE": trf_none,
    "TRF_FORCE_STRING": trf_force_string,
    "TRF_AMOUNT_SPACE_TO_DECIMAL": trf_amount_space_to_decimal,
    "TRF_DATE_DAYFIRST": trf_date_dayfirst,
    "TRF_BOOLEAN_SI_NO": trf_boolean_si_no,
    "TRF_INTEGER_STRICT": trf_integer_strict,
}
faltantes = set(df_map["transform"]) - set(TRANSFORMS)
assert not faltantes, f"Transformaciones sin implementar: {faltantes}"
print("Transformaciones disponibles:", ", ".join(TRANSFORMS))

Transformaciones disponibles: TRF_NONE, TRF_FORCE_STRING, TRF_AMOUNT_SPACE_TO_DECIMAL, TRF_DATE_DAYFIRST, TRF_BOOLEAN_SI_NO, TRF_INTEGER_STRICT


## 5. Selección de archivos por rango de fechas

La fecha de snapshot se extrae del nombre (`DD-MM-YYYY`, según `source.snapshot_date_extraction`). Los archivos con nombre inválido se omiten. La búsqueda es recursiva en `data/raw/cgr` (soporta subcarpetas por año).

In [20]:
PATRON_ARCHIVO = re.compile(mapping["source"]["snapshot_date_extraction"]["pattern"])


def listar_archivos(fecha_inicio: str, fecha_fin: str, raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    f_ini = datetime.strptime(str(fecha_inicio), "%Y%m%d").date()
    f_fin = datetime.strptime(str(fecha_fin), "%Y%m%d").date()
    if f_ini > f_fin:
        raise ValueError(f"FECHA_INICIO ({fecha_inicio}) no puede ser mayor que FECHA_FIN ({fecha_fin}).")

    registros = []
    for ruta in raw_dir.rglob("DataSet-Obras-Publicas*.xlsx"):
        m = PATRON_ARCHIVO.fullmatch(ruta.name)
        if not m:
            log.warning("Archivo omitido (nombre no coincide con el patrón): %s", ruta.name)
            continue
        dd, mm, yyyy = m.groups()
        try:
            fecha = datetime(int(yyyy), int(mm), int(dd)).date()
        except ValueError:
            log.warning("Archivo omitido (fecha inválida): %s", ruta.name)
            continue
        if f_ini <= fecha <= f_fin:
            registros.append({"fecha_foto": fecha, "nombre_archivo": ruta.name, "ruta": ruta})

    df = pd.DataFrame(registros, columns=["fecha_foto", "nombre_archivo", "ruta"])
    return df.sort_values("fecha_foto").reset_index(drop=True)


archivos = listar_archivos(FECHA_INICIO, FECHA_FIN)
print(f"Rango: {FECHA_INICIO} -> {FECHA_FIN} | Archivos encontrados: {len(archivos)}")
archivos[["fecha_foto", "nombre_archivo"]]

Rango: 20260201 -> 20260401 | Archivos encontrados: 3


,fecha_foto,nombre_archivo
0,2026-02-01,DataSet-Obras-Publicas 01-02-2026.xlsx
1,2026-03-01,DataSet-Obras-Publicas 01-03-2026.xlsx
2,2026-04-01,DataSet-Obras-Publicas 01-04-2026.xlsx


## 6. Carga de un archivo + aplicación del mapping

Se lee la hoja con `header=None` para poder resolver **encabezados duplicados por posición**. La fila de cabecera se autodetecta (fila que contiene `Código INFOBRAS`) o se usa `FILA_CABECERA`.

In [21]:
def detectar_fila_cabecera(ruta: Path, max_filas: int = 15) -> int:
    """Retorna la fila (1-based) de la cabecera buscando 'código infobras' en las primeras filas."""
    muestra = pd.read_excel(ruta, header=None, nrows=max_filas, dtype=str)
    for i in range(len(muestra)):
        valores = {norm_header(v) for v in muestra.iloc[i].dropna()}
        if "código infobras" in valores or "codigo infobras" in valores:
            return i + 1
    raise ValueError(f"No se encontró la fila de cabecera en {ruta.name}")


def cargar_archivo(ruta: Path, fecha_foto, fila_cabecera=None) -> pd.DataFrame:
    fila = fila_cabecera or detectar_fila_cabecera(ruta)
    crudo = pd.read_excel(ruta, header=None, skiprows=fila - 1, dtype=str)

    # Encabezados por posición: (nombre normalizado, nº de ocurrencia)
    encabezados = crudo.iloc[0].tolist()
    datos = crudo.iloc[1:].reset_index(drop=True)
    ocurrencias, claves = {}, []
    for h in encabezados:
        n = norm_header(h) if pd.notna(h) else ""
        ocurrencias[n] = ocurrencias.get(n, 0) + 1
        claves.append((n, ocurrencias[n]))

    posicion = {clave: i for i, clave in enumerate(claves)}

    # Columnas de origen sin mapping -> warning y se omiten
    sin_mapping = [encabezados[i] for i, c in enumerate(claves) if c not in LOOKUP]
    if sin_mapping:
        log.warning("%s: %d columna(s) de origen sin mapping (omitidas): %s", ruta.name, len(sin_mapping), sin_mapping)

    salida = {}
    for r in df_map.itertuples():
        pos = posicion.get((r.source_norm, r.occurrence))
        if pos is None:
            log.warning("%s: columna de origen ausente '%s' (ocurrencia %s) -> NULL", ruta.name, r.source, r.occurrence)
            serie = pd.Series([None] * len(datos), dtype="object")
        else:
            serie = datos.iloc[:, pos]
        salida[r.target] = TRANSFORMS[r.transform](serie)

    df = pd.DataFrame(salida)
    df = df.dropna(how="all").reset_index(drop=True)  # elimina filas completamente vacías

    # Columnas de auditoría: solo las marcadas active: true en el mapping
    contexto = {"fecha_foto": pd.Timestamp(fecha_foto), "nombre_archivo": ruta.name}
    for destino in AUDIT_ACTIVE:
        df[destino] = AUDIT_GENERATORS[destino](contexto)
    return df

## 7. Carga consolidada del rango

In [22]:
def cargar_rango(fecha_inicio: str, fecha_fin: str, fila_cabecera=None) -> pd.DataFrame:
    """Carga y consolida todos los snapshots entre fecha_inicio y fecha_fin (YYYYMMDD, inclusive)."""
    archivos = listar_archivos(fecha_inicio, fecha_fin)
    if archivos.empty:
        raise FileNotFoundError(f"No hay archivos en {RAW_DIR} para el rango {fecha_inicio}-{fecha_fin}")

    partes, resumen = [], []
    for fila in archivos.itertuples():
        log.info("Cargando %s ...", fila.nombre_archivo)
        try:
            df = cargar_archivo(fila.ruta, fila.fecha_foto, fila_cabecera)
        except Exception as e:  # reject_policy: log_and_continue
            log.error("Error cargando %s: %s", fila.nombre_archivo, e)
            resumen.append({"archivo": fila.nombre_archivo, "fecha_foto": fila.fecha_foto, "registros": 0, "estado": f"ERROR: {e}"})
            continue
        partes.append(df)
        resumen.append({"archivo": fila.nombre_archivo, "fecha_foto": fila.fecha_foto, "registros": len(df), "estado": "OK"})
        log.info("   -> %s registros", f"{len(df):,}")

    if not partes:
        raise RuntimeError("No se pudo cargar ningún archivo.")
    return pd.concat(partes, ignore_index=True), pd.DataFrame(resumen)


df_cgr, resumen_carga = cargar_rango(FECHA_INICIO, FECHA_FIN, FILA_CABECERA)
display(resumen_carga)
print(f"\nDataset consolidado: {df_cgr.shape[0]:,} registros x {df_cgr.shape[1]} columnas")

21:34:26 [INFO] Cargando DataSet-Obras-Publicas 01-02-2026.xlsx ...
21:37:18 [INFO]    -> 189,309 registros
21:37:18 [INFO] Cargando DataSet-Obras-Publicas 01-03-2026.xlsx ...
21:40:45 [INFO]    -> 190,111 registros
21:40:45 [INFO] Cargando DataSet-Obras-Publicas 01-04-2026.xlsx ...
21:44:28 [INFO]    -> 191,180 registros


,archivo,fecha_foto,registros,estado
0,DataSet-Obras-Publicas 01-02-2026.xlsx,2026-02-01,189309,OK
1,DataSet-Obras-Publicas 01-03-2026.xlsx,2026-03-01,190111,OK
2,DataSet-Obras-Publicas 01-04-2026.xlsx,2026-04-01,191180,OK



Dataset consolidado: 570,600 registros x 114 columnas


## 8. Validación rápida del dataset

In [23]:
# Conciliación: filas por snapshot vs. filas del DataFrame
conteo = df_cgr.groupby("fecha_foto").size().rename("filas_df").reset_index()
display(conteo.merge(resumen_carga.assign(fecha_foto=pd.to_datetime(resumen_carga["fecha_foto"])), on="fecha_foto"))

# Nulos de la clave de negocio (DQ001)
print("codigo_infobras nulos:", int(df_cgr["codigo_infobras"].isna().sum()))

# Tipos resultantes
print()
df_cgr.dtypes.astype(str).value_counts().to_frame("columnas")

,fecha_foto,filas_df,archivo,registros,estado
0,2026-02-01,189309,DataSet-Obras-Publicas 01-02-2026.xlsx,189309,OK
1,2026-03-01,190111,DataSet-Obras-Publicas 01-03-2026.xlsx,190111,OK
2,2026-04-01,191180,DataSet-Obras-Publicas 01-04-2026.xlsx,191180,OK


codigo_infobras nulos: 0



,columnas
string,48
Float64,21
Int64,18
datetime64[us],17
boolean,8
datetime64[s],2


In [24]:
pd.set_option('display.max_columns', None)

In [25]:
df_cgr.shape

(570600, 114)

In [26]:
df_cgr[df_cgr['codigo_infobras']=='121720']

,codigo_entidad,entidad_publica,codigo_infobras,nombre_obra,modalidad_ejecucion_obra,flg_corresponde_saldo_obra,codigo_infobras_obra_inicial,naturaleza_obra,tipo_obra_nivel_1,tipo_obra_nivel_2,tipo_obra_nivel_3,estado_ejecucion,flg_reconstruccion_con_cambios,flg_reactivacion_economica,otra_marca,n_informes_monitores_ciudadanos,n_denuncias_viculadas,n_informes_control,n_comentarios_ciudadanos,flg_obra_caracter_reservado,nivel_gobierno,sector_entidad,codigo_unico,codigo_snip,nombre_proyecto,estado_proyecto,tipo_formato,monto_viable_aprobado,fecha_aprobacion,monto_viable_actualizado,fecha_actualizacion,n_obras_relacionadas_proyecto,departamento,provincia,distrito,direccion_informacion_referencia,tipo_ubicacion,fecha_aprobacion_expediente,monto_aprobacion_et,moneda,tasa_cambio,monto_aprobado_soles,documento_aprobacion_expediente,ruc_1,razon_social_empresa_consorcio_1,monto_contrato_soles_1,ruc_2,razon_social_empresa_consorcio_2,monto_contrato_soles_2,tipo_documento_identidad,numero_documento_identidad,nombres_apellidos,colegiatura,numero_colegiatura,fecha_inicio_supervision,fecha_fin_supervision,tipo_documento_identidad_2,numero_documento_identidad_2,nombres_apellidos_2,colegiatura_2,numero_colegiatura_2,fecha_inicio_labores,fecha_fin_labores,fecha_inicio_obra,fecha_fin_programada_obra,plazo_ejecucion_dias,tipo_entrega,porcentaje_terreno_entregado,fecha_entrega_terreno,ano_avance,mes_avance,fecha_registro_avance,avance_fisico_programado_acumulado,avance_fisico_real_acumulado,monto_valorizacion_programado_acum,monto_valorizacion_ejecutado_acum,porcentaje_ejecucion_financiera,monto_ejecucion_financiera_obra,estado_actualizacion_registro_avance,comentarios,flg_existe_paralizacion,causal_paralizacion,fecha_paralizacion,numero_dias_paralizado,tipo_certificado_inversion_publica,numero_cipril_cipgn,fech_cipril_cipgn,monto_cipril_cipgn,n_modificaciones,n_dias_modificaciones_plazo,nuevo_plazo_ejecucion_dias,fecha_finalizacion_reprogramada_obra,n_controversias,n_adicionales_obra,monto_adicionales_obra_soles,n_adicionales_supervision,monto_adicionales_supervision_soles,n_deductivos_obra,monto_deductivo_obra_soles,n_deductivo_supervision,monto_deductivos_supervision_soles,fecha_finalizacion_real,motivo_caso_no_llegue_100_porciento,flg_tiene_recepcion_total,fecha_recepcion,flg_tiene_liquidacion_obra,fecha_aprobacion_liquidacion_obra,costo_obra_soles,flg_corresponde_transferencia_otra_entidad,fecha_transferencia,tipo_entidad_recepciona_obra,ano_primer_devengado,monto_total_devengado_proyecto,fecha_foto
101802,6012,PROGRAMA NACIONAL DE INFRAESTRUCTURA EDUCATIVA...,121720,ELABORACIÓN DE EXPEDIENTE TÉCNICO Y EJECUCIÓN ...,Contrata,<NA>,<NA>,Reconstrucción,Educación/Cultura,Educación Básica,Centro Educativo/Colegio,Finalizado,True,<NA>,<NA>,0,<NA>,1,0,<NA>,GOBIERNO NACIONAL,EDUCACIÓN,2458617,2458617,<NA>,<NA>,<NA>,<NA>,NaT,0.0,NaT,0,ANCASH,ANTONIO RAYMONDI,CHINGAS,Chahuarcon,<NA>,2023-03-23,4435631.37,S/ Soles,1.0,4435631.37,Resolución Jefatural 035 2023 UGRD,C0003403714,CONSORCIO CHINGAS,4435631.37,10426481915,ASENCIO VASQUEZ GABRIEL ERNESTO,256141.15,D N I,16756314,HUGO PABEL CUBAS CHAVARRY,C I P,8062,2022-09-08,2023-09-09,D N I,22513678,LUDWIG DAVID SALAZAR SANCHEZ,C I P,12863,2023-08-04,2023-09-09,2023-04-13,2023-09-10,150,Total,100.0,2022-09-26,2025,3,2025-04-28,80.8,16.38,3477101.1,704935.21,16.38,704935.21,Avance Actualizado,Paralizada,True,Otras causales,2023-09-01,30,<NA>,<NA>,NaT,<NA>,0,0,150,2023-09-10,0,0,0.0,0,0.0,0,0.0,0,0.0,2025-05-07,Resolución de contrato,True,2025-05-13,<NA>,NaT,<NA>,<NA>,NaT,<NA>,2020,1554888.4,2026-02-01
291045,6012,PROGRAMA NACIONAL DE INFRAESTRUCTURA EDUCATIVA...,121720,ELABORACIÓN DE EXPEDIENTE TÉCNICO Y EJECUCIÓN ...,Contrata,<NA>,<NA>,Reconstrucción,Educación/Cultura,Educación Básica,Centro Educativo/Colegio,Finalizado,True,<NA>,<NA>,0,<NA>,1,0,<NA>,GOBIERNO NACIONAL,EDUCACIÓN,2458617,2458617,<NA>,<NA>,<NA>,<NA>,NaT,0.0,NaT,0,ANCASH,ANTONIO RAYMONDI,CHINGAS,Chahuarcon,<NA>,2023-03-23,4435631.37,S/ Soles,1.0

## 9. Exportación opcional a CSV (`data/stg`)

Activar con `EXPORTAR_CSV = True` en la celda de parámetros (o ejecutar `exportar_csv(df_cgr)` manualmente). Codificación `utf-8-sig` (compatible con Excel), separador `,` y decimal `.`.

In [27]:
def exportar_csv(df: pd.DataFrame, nombre: str | None = None) -> Path:
    STG_DIR.mkdir(parents=True, exist_ok=True)
    nombre = nombre or f"sa_obras_publicas_infobras_{FECHA_INICIO}_{FECHA_FIN}.csv"
    ruta = STG_DIR / nombre
    df.to_csv(ruta, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")#, compression='gzip')
    log.info("CSV exportado: %s (%s registros, %.1f MB)", ruta, f"{len(df):,}", ruta.stat().st_size / 1024**2)
    return ruta


if EXPORTAR_CSV:
    ruta_csv = exportar_csv(df_cgr, NOMBRE_CSV)
else:
    print("Exportación desactivada (EXPORTAR_CSV = False).")

21:45:00 [INFO] CSV exportado: C:\ws\formacion\uni-mia\mia-tesis-opa\data\stg\sa_obras_publicas_infobras_20260201_20260401.csv (570,600 registros, 577.9 MB)
